In [15]:
from datetime import datetime, timedelta
from finpie.datasource.service import DataService
from mt5_backtest_runner import run_mt5_backtest
import glob
import os
import pandas as pd
from functools import reduce
import re
import math
from finpie.data.timeseries import TimeSeries
from finpie.data.multitimeseries import MultiTimeSeries
from finpie.data.spreadtimeseries import SpreadTimeSeries
from finpie.data.ratiotimeseries import RatioTimeSeries
import numpy as np
import matplotlib.pyplot as plt
from finpie.analytics.genetic_portfolio_optimizer import GeneticPortfolioOptimizer, OptimizationConfig

In [8]:
opt_path = r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization"

In [12]:
# Get all files in the prod directory
opt_files = glob.glob(os.path.join(opt_path, '*'))

In [13]:
backtested = [r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_bova11_ma_ma_distance_spread.set",
r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_bova11_momentum_ratio.set",
r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_bova11_prices_ratio.set",
r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_bova11_price_ma_distance_spread.set",
r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_bova11_returns_spread.set",
r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_bova11_rsi_ratio.set",
r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_ma_ma_distance.set",
r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_momentum.set",
r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_momentum_mama_distance_spread.set",
r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_momentum_price_ma_distance_spread.set",
r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_momentum_returns_ratio.set",
r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_prices.set",
r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_prices_mama_distance_spread.set",
r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_prices_momentum_ratio.set",
r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_prices_price_ma_distance_spread.set",
r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_prices_returns_ratio.set",
r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_prices_rsi_ratio.set",
r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_price_ma_dist.set",
r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_returns.set",
r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_returns_mama_distance_spread.set",
r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_returns_price_ma_distance_spread.set",
r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_rsi.set",
r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_rsi_mama_distance_spread.set",
r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_rsi_momentum_ratio.set",
r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_rsi_price_ma_distance_spread.set",
r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_rsi_returns_ratio.set",
r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_wdo_ma_ma_distance_spread.set",
r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_wdo_momentum_ratio.set",
r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_wdo_prices_ratio.set",
r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_wdo_price_ma_distance_spread.set",
r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_wdo_returns_spread.set",
r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_wdo_rsi_ratio.set",
r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_wsp_ma_ma_distance_spread.set",
r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_wsp_momentum_ratio.set",
r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_wsp_prices_ratio.set",
r"C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_wsp_price_ma_distance_spread.set"]

In [14]:
for file in opt_files:
    print(file)
    if file not in backtested:
        run_mt5_backtest(
            expert_name='sino\\ZScoreStrategy',
            expert_params_file=f"sino\\optimization\\{file.split('\\')[-1]}",
            symbol="WIN$N",
            start_date="2020.01.01",
            end_date="2025.12.31",
            timeframe="M1",
            model=2,
            optimization=True
        )

C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_bova11_ma_ma_distance_spread.set
C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_bova11_momentum_ratio.set
C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_bova11_prices_ratio.set
C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_bova11_price_ma_distance_spread.set
C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_bova11_returns_spread.set
C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF550E51FF075\MQL5\Profiles\Tester\sino\optimization\win_bova11_rsi_ratio.set
C:\Users\enzob\AppData\Roaming\MetaQuotes\Terminal\D0E8209F77C8CF37AD8BF